<h1>Factory Machine Status</h1>

<hr/>

<p>Name: <strong>Goh Kun Ming</strong><br/></p>
<p>School: <strong>Singapore Polytechnic, School of Computing</strong><br/></p>
<p>Diploma: <strong>Diploma in Applied AI &amp; Analytics</strong><br/></p>
<p>Module: <strong>AI &amp; Machine Learning (ST1511)</strong><br/></p>
<p>Assessment: <strong>CA1 Part A</strong><br/></p>
<p>Academic Period: <strong>AY24/25 Year 1 Semester 2</strong><br/></p>
<p>Lecturer: <strong>Adjunct Lecturer Tai Hock Lin (Andy)</strong><br/></p>

<hr/>

<h1>Notebook Objective</h1>

<hr/>

<p>The objective of this notebook is to demonstrate how a trained model is used for prediction. I will train a small temporary model, score holdout rows, and interpret the prediction output format.</p>

<p>This notebook is part of the broken down notebook workflow. The original CA1 notebook is still maintained as <code>00_original_ca1_submission.ipynb</code>, while this notebook keeps the same report-style explanation and interpretation format in a smaller, easier-to-review file.</p>

<p>Notebook: <strong>04 Evaluation And Prediction</strong></p>


<hr/>
<h1>1.&nbsp;&nbsp;&nbsp;&nbsp;Importing of Modules</h1>
<hr/>

<p>Within this section, I will import the utilities needed to train a temporary model and generate predictions. This notebook focuses on the model output contract rather than full model development.</p>


<h2>1.1&nbsp;&nbsp;&nbsp;&nbsp;Importing Prediction Utilities</h2>

<p>The following cell imports the data loader, training function, probability scoring function, and project constants.</p>


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from fault_prediction.config import DEFAULT_DATA_PATH, RANDOM_STATE, TARGET_COLUMN
from fault_prediction.data import load_factory_data
from fault_prediction.models import predict_scores, train_model


<p><strong>Interpretation:</strong> The prediction workflow uses the same production pipeline as training. This means feature engineering, imputation, scaling, encoding, and prediction are handled consistently.</p>


<hr/>
<h1>2.&nbsp;&nbsp;&nbsp;&nbsp;Preparing Model and Holdout Data</h1>
<hr/>

<p>In this section, I will create a small training sample and keep the remaining rows as holdout data for prediction demonstration.</p>


<h2>2.1&nbsp;&nbsp;&nbsp;&nbsp;Training a Temporary Demonstration Model</h2>

<p>The model trained here is only for notebook demonstration. Production model artifacts should be generated through the CLI and stored outside Git.</p>


In [ ]:
df = load_factory_data(DEFAULT_DATA_PATH)
train_df, holdout_df = train_test_split(
    df,
    train_size=1000,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)

result = train_model(train_df, profile='fast', n_jobs=1)
artifact = result.artifact
pd.Series(result.metrics, name='Validation Metric')


<p><strong>Interpretation:</strong> A temporary model has been trained and evaluated. The selected threshold is stored in the artifact and will be used when converting probabilities into predicted machine status values.</p>


<hr/>
<h1>3.&nbsp;&nbsp;&nbsp;&nbsp;Prediction</h1>
<hr/>

<p>In this section, I will generate predictions for holdout rows. The model produces a probability of fault, and the saved threshold converts that probability into a class label.</p>


<h2>3.1&nbsp;&nbsp;&nbsp;&nbsp;Scoring Holdout Rows</h2>

<p>The next cell scores the first 20 holdout rows and displays the key prediction outputs.</p>


In [ ]:
model = artifact['model']
threshold = artifact['threshold']

preview_rows = holdout_df.head(20).copy()
X_preview = preview_rows.drop(columns=[TARGET_COLUMN])
probabilities = predict_scores(model, X_preview)
predictions = (probabilities >= threshold).astype(int)

prediction_preview = preview_rows[['Unique ID', 'Product ID', TARGET_COLUMN]].copy()
prediction_preview['predicted_probability_fault'] = probabilities
prediction_preview['predicted_machine_status'] = predictions
prediction_preview


<p><strong>Interpretation:</strong> The prediction output includes the probability of fault and the final predicted machine status. Keeping identifiers in the preview helps trace predictions back to the original rows, but identifiers are not used as model features.</p>


<hr/>
<h2>3.2&nbsp;&nbsp;&nbsp;&nbsp;Reviewing Prediction Summary</h2>

<p>The next cell summarises the predicted machine status values for the preview rows. This is a simple check to understand how many rows were classified as normal or abnormal.</p>


In [ ]:
prediction_preview['predicted_machine_status'].value_counts().sort_index().to_frame('Count')


<p><strong>Interpretation:</strong> The prediction summary gives a quick view of model decisions for the preview rows. In a real deployment, these predictions should be monitored over time and compared against confirmed machine outcomes when labels become available.</p>


<hr/>
<h1>4.&nbsp;&nbsp;&nbsp;&nbsp;Notebook Summary</h1>
<hr/>

<p>This notebook demonstrates the model prediction contract. For production-style prediction using a saved artifact, run the command below from the repository root:</p>

<pre><code>fault-predict predict --model models/fault_voting_classifier.joblib --input data/raw/factory_data.csv</code></pre>
